# Đề tài 10 · Nhận dạng mẫu: Phân loại Âm thanh & Phát hiện Giọng nói Nhân bản Zero-Shot
## Thực nghiệm 7.0: Khảo sát Đối sánh Đa Thuật toán Phân loại Phát hiện Âm thanh Giả mạo (T-A Benchmark)

**Nhóm nghiên cứu: Nhóm 05 · Chuyên ngành Khoa học Máy tính / Trí tuệ Nhân tạo**

### 1. Mục tiêu Khoa học
Tiến hành thực nghiệm đối chuẩn (benchmark) mở rộng giữa nhiều họ thuật toán học máy khác nhau trên cùng một phân vùng dữ liệu và không gian đặc trưng 75 chiều của bài toán T-A (phát hiện âm thanh thật / giả mạo):
- Khảo sát tương quan hiệu năng giữa các trường phái mô hình: Mô hình biên tối đa (SVM RBF), Mô hình tập hợp (Random Forest), Mô hình láng giềng gần nhất ($k$-Nearest Neighbors), và Mô hình xác suất thống kê (Gaussian Naive Bayes).
- Phân tích độ phức tạp tính toán, thời gian suy luận và khả năng khái quát hóa trên tập kiểm thử độc lập 7 người nói.

### 2. Thiết kế Thực nghiệm Đối chuẩn
- **Không gian thuộc tính:** 75 chiều đặc trưng âm học ($F_1, F_2, F_3$) được chuẩn hóa theo z-score (StandardScaler).
- **Phân hoạch dữ liệu:** Giữ nguyên cấu trúc phân chia 18 người huấn luyện (Train) / 7 người kiểm thử độc lập (Test) đã thiết lập ở Thực nghiệm 3.0.
- **Thuật toán so sánh:**
  1. *SVM (Radial Basis Function Kernel):* Mô hình tối ưu biên phi tuyến.
  2. *Random Forest (100 Decision Trees):* Phương pháp tập hợp bagging giảm phương sai.
  3. *$k$-NN ($k=5$ Minkowski distance):* Phương pháp phân loại dựa trên khoảng cách hình học.
  4. *Gaussian Naive Bayes:* Mô hình xác suất tiên đề phân phối chuẩn.

### 3. Tiêu chí Đánh giá
- Độ chính xác tổng thể (Accuracy), Độ nhạy (Recall), F1-Score vĩ mô (Macro F1).
- Diện tích dưới đường cong ROC (ROC-AUC) và Ma trận nhầm lẫn đối sánh.
- Xuất bản bảng tổng hợp chỉ số kỹ thuật và báo cáo so sánh trực quan.


In [ ]:
# Bước 7: so sánh bổ sung các mô hình thật/giả trên đặc trưng T-A đã lưu.
from pathlib import Path
import os, json, hashlib, base64, platform, zipfile
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import sklearn
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score,balanced_accuracy_score,precision_recall_fscore_support,confusion_matrix,roc_auc_score
import joblib

SEED=20261103
NAMES=['manifest_TA.csv','dac_trung_TA.csv','CV_theo_nguoi.csv','du_doan_test_TA.csv','ket_qua_TA.csv']
roots=[Path('/kaggle/input'),Path('/home/user/uploads'),Path.cwd()]
def find_file(name):
    hits=[]
    for root in roots:
        if root.exists():
            hits.extend(p for p in root.rglob(name) if p.is_file() and 'buoc7_ket_qua' not in str(p))
    hits=sorted({p.resolve() for p in hits},key=str)
    if not hits: raise FileNotFoundError('Thiếu '+name+'; gắn đầy đủ dữ liệu Bước 3.')
    sha={hashlib.sha256(p.read_bytes()).hexdigest() for p in hits}
    if len(sha)>1: raise RuntimeError('Có nhiều tệp '+name+' khác nội dung; gắn đúng 1 bộ Bước 3: '+str(hits))
    return hits[0]
paths={n:find_file(n) for n in NAMES}
print('Đầu vào:',*(str(x) for x in paths.values()),sep='\n  ')
base=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/home/user/uploads')
out=base/'buoc7_ket_qua';out.mkdir(parents=True,exist_ok=True)
m=pd.read_csv(paths['manifest_TA.csv']); f=pd.read_csv(paths['dac_trung_TA.csv']); oldcv=pd.read_csv(paths['CV_theo_nguoi.csv']); oldpred=pd.read_csv(paths['du_doan_test_TA.csv']); oldres=pd.read_csv(paths['ket_qua_TA.csv'])
assert m.id_clip.is_unique and f.id_clip.is_unique and set(m.id_clip)==set(f.id_clip)
d=m.merge(f,on='id_clip',validate='one_to_one'); cols=sorted(c for c in f if c.startswith(('F1_','F2_','F3_')))
assert len(cols)==75 and np.isfinite(d[cols].to_numpy(float)).all()
assert len(d)==1480 and set(d.nguon)=={'that','omnivoice','omdiom','vieneu'}
assert (d.groupby('pair_id').size()==4).all() and (d.groupby('pair_id').speaker.nunique()==1).all()
tr=d[d.tap=='train'].copy();te=d[d.tap=='test'].copy();be=te[te.test_can_bang.astype(bool)].copy()
assert len(tr)==1060 and len(te)==420 and len(be)==210 and set(tr.speaker).isdisjoint(set(te.speaker))
assert tr.speaker.nunique()==18 and te.speaker.nunique()==7 and set(tr.pair_id).isdisjoint(set(te.pair_id))
assert be.pair_id.nunique()==105 and (be.groupby('pair_id').nhan.agg(['size','sum'])==[2,1]).all().all()
X=tr[cols].to_numpy(float); y=tr.nhan.to_numpy(int)
folds=list(GroupKFold(n_splits=3).split(tr,groups=tr.speaker))
assert all(set(tr.iloc[a].speaker).isdisjoint(set(tr.iloc[b].speaker)) for a,b in folds)
# Cố định trước: các mô hình gốc giữ cấu hình Bước 3; 2 mô hình mới không dò siêu tham số.
constructors={
 'SVM':lambda:make_pipeline(StandardScaler(),SVC(C=1,kernel='rbf',class_weight='balanced')),
 'RandomForest':lambda:RandomForestClassifier(n_estimators=150,min_samples_leaf=2,class_weight='balanced',n_jobs=1,random_state=SEED),
 'k-NN':lambda:make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=5)),
 'GaussianNB':lambda:make_pipeline(StandardScaler(),GaussianNB(priors=[0.5,0.5]))}
cv=[]
for name,create in constructors.items():
 for k,(a,b) in enumerate(folds,1):
  mod=create();mod.fit(X[a],y[a]);pred=mod.predict(X[b]);
  cv.append({'mo_hinh':name,'fold':k,'balanced_accuracy':balanced_accuracy_score(y[b],pred),'macro_f1':precision_recall_fscore_support(y[b],pred,labels=[0,1],average='macro',zero_division=0)[2],'nguoi_train':tr.iloc[a].speaker.nunique(),'nguoi_val':tr.iloc[b].speaker.nunique()})
cv=pd.DataFrame(cv)
check=cv.merge(oldcv,on=['mo_hinh','fold'],suffixes=('_moi','_cu'),validate='one_to_one')
assert len(check)==6
for col in ['balanced_accuracy','macro_f1']:
 if not np.allclose(check[col+'_moi'],check[col+'_cu'],atol=1e-10,rtol=0): raise ValueError('CV cũ không tái hiện: '+col+'; dừng thay vì so sánh sai.')
summary=cv.groupby('mo_hinh',as_index=False).agg(cv_balanced_accuracy=('balanced_accuracy','mean'),cv_macro_f1=('macro_f1','mean')).sort_values(['cv_balanced_accuracy','mo_hinh'],ascending=[False,True])
print('\nCV 3 fold theo speaker:');print(summary.to_string(index=False))
# Hai bộ sung được đánh giá trên test cũ đã xem từ trước; không chọn lại mô hình ban đầu.
results=[];pred_rows=[];source_rows=[];matrices={}
for name,create in constructors.items():
 mod=create();mod.fit(X,y)
 if name=='SVM': joblib.dump({'model':mod,'feature_names':cols,'labels':{'0':'that','1':'gia'},'sr':16000,'luu_y':'Tái huấn luyện với đặc trưng cũ; so khớp dự đoán với Bước 3. Không phải xác thực thực tế.'},out/'svm_TA_tai_tao.joblib')
 for group,data in [('toan_bo_1_that_3_gia',te),('can_bang_1_that_1_gia',be)]:
  XX=data[cols].to_numpy(float); yy=data.nhan.to_numpy(int); pred=mod.predict(XX)
  # Điểm được dùng để xếp hạng, không trình bày như xác suất tin cậy.
  score=mod.decision_function(XX) if hasattr(mod,'decision_function') else mod.predict_proba(XX)[:,list(mod.classes_).index(1)]
  pr,re,f1,_=precision_recall_fscore_support(yy,pred,labels=[0,1],average='macro',zero_division=0)
  c=confusion_matrix(yy,pred,labels=[0,1]); matrices[(name,group)]=c.tolist()
  results.append({'mo_hinh':name,'tap_test':group,'so_clip':len(data),'accuracy':accuracy_score(yy,pred),'balanced_accuracy':balanced_accuracy_score(yy,pred),'precision_macro':pr,'recall_macro':re,'macro_f1':f1,'roc_auc':roc_auc_score(yy,score),'that_dung':int(c[0,0]),'that_nham_gia':int(c[0,1]),'gia_nham_that':int(c[1,0]),'gia_dung':int(c[1,1])})
  p=data[['id_clip','pair_id','speaker','cau_id','nguon','nhan']].copy();p['mo_hinh']=name;p['tap_test']=group;p['du_doan']=pred;p['diem_xep_hang_gia']=score;pred_rows.append(p)
  if group=='toan_bo_1_that_3_gia':
   for source,x in p[p.nhan==1].groupby('nguon'):
    source_rows.append({'mo_hinh':name,'bo_sinh':source,'so_clip':len(x),'dung_gia':int((x.du_doan==1).sum()),'recall_gia':float((x.du_doan==1).mean())})
results=pd.DataFrame(results);preds=pd.concat(pred_rows,ignore_index=True);sources=pd.DataFrame(source_rows)
orig=oldpred[oldpred.tap_test=='toan_bo_1_that_3_gia'][['id_clip','du_doan']]
current=preds[(preds.mo_hinh=='SVM')&(preds.tap_test=='toan_bo_1_that_3_gia')][['id_clip','du_doan']]
comp=orig.merge(current,on='id_clip',suffixes=('_cu','_tai_tao'),validate='one_to_one')
assert len(comp)==420 and (comp.du_doan_cu==comp.du_doan_tai_tao).all(), 'SVM dự đoán không khớp Bước 3; dừng.'
for group in ['toan_bo_1_that_3_gia','can_bang_1_that_1_gia']:
 a=results[(results.mo_hinh=='SVM')&(results.tap_test==group)].iloc[0];b=oldres[oldres.tap_test==group].iloc[0]
 for col in ['accuracy','balanced_accuracy','precision_macro','recall_macro','macro_f1','roc_auc']:
  assert np.isclose(a[col],b[col],atol=1e-9,rtol=0),f'Kết quả SVM {col} không khớp'
print('\nTest đầy đủ (1 thật / 3 giả) – so sánh thăm dò:');print(results[results.tap_test=='toan_bo_1_that_3_gia'].to_string(index=False))
print('\nĐã tái hiện chính xác 420/420 dự đoán test của SVM Bước 3.')
for name,frame in [('CV_theo_nguoi_bo_sung.csv',cv),('tong_hop_CV_train.csv',summary),('so_sanh_test_cu.csv',results),('du_doan_cac_mo_hinh.csv',preds),('recall_gia_theo_bo_sinh.csv',sources)]: frame.to_csv(out/name,index=False,encoding='utf-8-sig')
info={'muc_tieu':'So sanh bo sung knn va GaussianNB cho T-A that/gia, du lieu cu','du_lieu':'370 cap, 18 speaker train, 7 speaker test','test_day_du':420,'test_can_bang_la_tap_con':210,'phan_chia_cv':'GroupKFold 3 fold tren 18 speaker train','cv_cu_so_khop':len(check),'SVM_test_so_khop':int(len(comp)),'mo_hinh_chinh_da_chon_tu_Buoc3':'SVM','dac_trung':cols,'cau_hinh':{'SVM':'RBF C=1 class_weight=balanced; StandardScaler','RandomForest':'150 trees, min_samples_leaf=2, class_weight=balanced, seed=20261103','k-NN':'k=5, Euclidean, StandardScaler; không class_weight','GaussianNB':'priors=[0.5,0.5], StandardScaler'},'nguon_sha256':{k:hashlib.sha256(v.read_bytes()).hexdigest() for k,v in paths.items()},'versions':{'python':platform.python_version(),'scikit_learn':sklearn.__version__,'numpy':np.__version__,'pandas':pd.__version__},'han_che':['test va so lieu da xem truoc; so sanh bo sung mang tinh tham do','OmDiOm fine-tune VIVOS, chua doi chieu trung lap tung clip','7 speaker test it va 5 cap loi bi loai o train','balanced test la tap con, khong phai test doc lap','resample khong xoa het dau vet bo sinh va kenh thu']}
(out/'thong_tin_buoc7.json').write_text(json.dumps(info,ensure_ascii=False,indent=2),encoding='utf8')
plt.rcParams.update({'font.family':'DejaVu Sans','axes.spines.top':False,'axes.spines.right':False,'figure.dpi':145})
fig,ax=plt.subplots(figsize=(8.5,4.2)); x=summary.sort_values('cv_balanced_accuracy');bars=ax.barh(x.mo_hinh,x.cv_balanced_accuracy*100,color=['#597ca1','#8d72b8','#e99a50','#207e9c']);ax.bar_label(bars,fmt='%.2f%%',padding=3);ax.set(xlim=(0,105),xlabel='Balanced accuracy trung bình (%)',title='CV theo người · chỉ trên 18 người train');fig.tight_layout();fig.savefig(out/'01_cv_train.png',bbox_inches='tight');plt.close(fig)
fig,ax=plt.subplots(figsize=(8.5,4.4));r=results[results.tap_test=='toan_bo_1_that_3_gia']; bars=ax.bar(r.mo_hinh,r.balanced_accuracy*100,color=['#207e9c','#597ca1','#e99a50','#8d72b8']);ax.bar_label(bars,fmt='%.2f%%',padding=3);ax.set(ylim=(0,110),ylabel='Balanced accuracy (%)',title='Test cũ 1 thật : 3 giả · so sánh bổ sung');fig.tight_layout();fig.savefig(out/'02_test_day_du.png',bbox_inches='tight');plt.close(fig)
fig,axs=plt.subplots(1,4,figsize=(13,3.8));
for ax,name in zip(axs,constructors):
 cm=np.array(matrices[(name,'toan_bo_1_that_3_gia')]);im=ax.imshow(cm,cmap='Blues',vmin=0,vmax=315)
 for i in range(2):
  for j in range(2):ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>170 else '#17263d',fontsize=13,fontweight='bold')
 ax.set(xticks=[0,1],xticklabels=['Thật','Giả'],yticks=[0,1],yticklabels=['Thật','Giả'],title=name,xlabel='Dự đoán');
axs[0].set_ylabel('Nhãn thực');fig.suptitle('Ma trận nhầm lẫn · test 420 clip',fontweight='bold');fig.tight_layout();fig.savefig(out/'03_ma_tran.png',bbox_inches='tight');plt.close(fig)
def embed(name):return 'data:image/png;base64,'+base64.b64encode((out/name).read_bytes()).decode()
def table(x):return x.to_html(index=False,border=0,float_format=lambda v:f'{v:.4f}',classes='data')
full=results[results.tap_test=='toan_bo_1_that_3_gia'].drop(columns=['tap_test']);bal=results[results.tap_test=='can_bang_1_that_1_gia'].drop(columns=['tap_test']);
html=f'''<!doctype html><html lang="vi"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Bước 7 · So sánh mô hình thật/giả</title><style>body{{font-family:system-ui,Arial,sans-serif;background:#f2f6fb;color:#17263d;line-height:1.65;margin:0}}main{{max-width:1120px;margin:auto;padding:32px}}header,section{{border-radius:16px;background:white;padding:24px;margin-bottom:18px;box-shadow:0 4px 22px #12345410}}header{{background:#15345d;color:white}}h2{{color:#184d86}}h1,h2{{line-height:1.25}}img{{max-width:100%;height:auto}}table{{border-collapse:collapse;width:100%;font-size:13px}}th,td{{padding:8px;border-bottom:1px solid #dce4ef;text-align:left;white-space:nowrap}}th{{background:#edf3fb}}.scroll{{overflow:auto}}.warning{{border-left:5px solid #e6a348;background:#fff8eb;padding:14px}}small{{color:#4b5f78}}</style></head><body><main><header><small style="color:#c5dffb">THÍ NGHIỆM BỔ SUNG · KHÔNG SINH LẠI AUDIO</small><h1>Bước 7: so sánh mô hình thật/giả</h1><p>Giữ nguyên phân chia T-A · 18 người train, 7 người test · 75 đặc trưng</p></header>
<section><h2>1. Thiết kế và kiểm chứng</h2><p>SVM RBF C=1 và Random Forest giữ cấu hình cũ. Bổ sung k-NN k=5 và Gaussian NB với prior cân bằng, không dò siêu tham số. StandardScaler được đặt trong pipeline của SVM, k-NN và Gaussian NB. GroupKFold 3 fold theo speaker như Bước 3.</p><p>Kiểm tra: 6/6 kết quả CV SVM và Random Forest tái hiện; SVM cho cùng nhãn trên 420/420 clip test như tệp Bước 3. Các điểm cũ không bị thay thế.</p><div class="warning"><b>Đây là so sánh bổ sung, mang tính thăm dò:</b> test đã được xem trước khi thêm mô hình; không coi là đánh giá độc lập để khẳng định mô hình thắng hay thay lại mô hình T-A gốc.</div></section>
<section><h2>2. CV trên train</h2><div class="scroll">{table(summary)}</div><img src="{embed('01_cv_train.png')}" alt="CV theo người"><p>SVM là mô hình đã chọn ở Bước 3 khi chỉ so sánh SVM với Random Forest; không chọn lại dựa trên test đã xem.</p></section>
<section><h2>3. Test đầy đủ: 105 thật, 315 giả</h2><div class="scroll">{table(full)}</div><img src="{embed('02_test_day_du.png')}" alt="Balanced accuracy test"><img src="{embed('03_ma_tran.png')}" alt="Bốn ma trận nhầm lẫn"><p>Ma trận: hàng là nhãn thật, cột là dự đoán. <code>that_nham_gia</code> là báo nhầm clip thật là giả; <code>gia_nham_that</code> là bỏ lọt clip giả. Do lệch lớp, xem cả balanced accuracy và macro-F1 thay vì chỉ accuracy.</p></section>
<section><h2>4. Test cân bằng 105 thật + 105 giả</h2><div class="scroll">{table(bal)}</div><p>Đây là <b>tập con</b> của test đầy đủ, không phải tập test độc lập thứ hai. Một clip giả được chọn theo vòng luân phiên bộ sinh cho từng câu.</p></section>
<section><h2>5. Độ nhạy phát hiện giả theo bộ sinh</h2><div class="scroll">{table(sources)}</div><p>Độ nhạy trên từng bộ sinh không thay thế phân tích báo nhầm giọng thật. Điểm xếp hạng trong CSV không được coi là xác suất đã hiệu chuẩn.</p></section>
<section><h2>6. Giới hạn</h2><ul><li>Các mô hình bổ sung được thử sau khi đã xem dữ liệu và kết quả test; không dùng test này để đưa ra kết luận so sánh xác nhận độc lập.</li><li>Chỉ 7 người test; clip cùng người, cùng câu có tương quan. Không coi 420 clip là 420 người độc lập.</li><li>OmDiOm đã fine-tune VIVOS; chưa xác minh trùng lặp từng ID với tập huấn luyện TTS. Test theo người chỉ độc lập đối với bộ phân loại.</li><li>5 cặp sinh thất bại đã loại đồng bộ khỏi train; phân phối chọn lọc, lỗi đọc của OmniVoice, codec và kênh thu có thể ảnh hưởng kết quả.</li><li>Không khẳng định phát hiện được mọi giọng giả hoặc dùng dự đoán để xác thực danh tính trong thực tế.</li></ul></section></main></body></html>'''
(out/'BAO_CAO_BUOC7.html').write_text(html,encoding='utf8')
zip_path=base/'KET_QUA_BUOC7_SO_SANH_TA.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
 for p in sorted(out.iterdir()):
  if p.is_file():z.write(p,p.name)
print('\nBáo cáo:',out/'BAO_CAO_BUOC7.html','\nZIP:',zip_path)
